# BigAlpha 2026 - Factor v454

## `crowd_residual_indneut_barra_v454`

**Crowd-Residual -> IndNeut -> BARRA (3-Stage Pipeline)**

- Data: bar1m (tail) + exposure (BARRA10, industry_level1_code)
- Step1: Remove crowd (BTOP crowding) from tail signal
- Step2: Cross-sectional standardize
- Step3: Within-industry neutralize
- Step4: Apply conviction threshold
- Step5: BARRA + industry neutralization (bc)
- Step6: Final crz
- Key insight: Apply neutralizations in a specific order — first anti-crowd, then industry, then BARRA
- Each stage adds a different type of purification — order matters for the interaction effects

In [ ]:
def main(datasources, start_date, end_date):
    import pandas as pd, numpy as np, dai
    bar1m = datasources["bar1m"]
    exp_table = datasources.get("exposure", "bigalpha_2026_exposure")
    eps = 1e-9
    BARRA = ['SIZE','BETA','MOMENTUM','RESVOL','SIZENL','BTOP','LIQUIDTY','EARNYILD','GROWTH','LEVERAGE']

    sql_bar = f"""SELECT CAST(strftime(date,'%Y-%m-%d') AS DATETIME) AS date, instrument,
        MIN(low) AS low, MAX(high) AS high,
        MIN_BY(open, date) AS o, MAX_BY(close, date) AS c,
        SUM(amount)/NULLIF(SUM(volume),0) AS vwap
    FROM {bar1m} WHERE date>='{start_date}' AND date<='{end_date}'
    GROUP BY CAST(strftime(date,'%Y-%m-%d') AS DATETIME), instrument"""
    d = dai.query(sql_bar).df()
    d['date'] = pd.to_datetime(d['date'])
    for c in ['low','high','o','c','vwap']:
        d[c] = pd.to_numeric(d[c], errors='coerce').astype('float64')
    d['tail'] = (d['vwap'] - d['c']) / (d['high'] - d['low'] + eps)

    sql_exp = f"SELECT date, instrument, {','.join(BARRA)}, industry_level1_code FROM {exp_table} WHERE date>='{start_date}' AND date<='{end_date}'"
    de = dai.query(sql_exp).df()
    de['date'] = pd.to_datetime(de['date'])
    for c in BARRA:
        de[c] = pd.to_numeric(de[c], errors='coerce').astype('float64')
    de['industry_level1_code'] = de['industry_level1_code'].astype(str)

    d = d.merge(de, on=['date','instrument'], how='inner')

    def crz(s):
        r = s.rank(pct=True) - 0.5; m = r.mean(); sd = r.std()
        return (r - m) / (sd + eps) if sd and sd > 0 else r * 0
    def xr(s):
        return s.rank(pct=True) - 0.5

    def bc(df, sc):
        df['_r'] = np.nan
        for dt, grp in df.groupby('date'):
            if len(grp) < 100: continue
            y = grp[sc]; my = y.notna()
            if my.sum() < 100: continue
            Xc = [c for c in BARRA if c in grp.columns]
            idm = pd.get_dummies(grp['industry_level1_code'].astype(str), prefix='x').astype(float)
            X = pd.concat([grp[Xc].fillna(0), idm], axis=1)
            m = my & X.notna().all(axis=1)
            if m.sum() < 100: continue
            Xs, ys = X[m].values, y[m].values
            try:
                beta, _, _, _ = np.linalg.lstsq(Xs, ys, rcond=None)
                df.loc[m[m].index, '_r'] = ys - Xs @ beta
            except: continue
        return df['_r']

    for col in ['tail'] + BARRA:
        d[col] = d[col].replace([np.inf,-np.inf],np.nan)
        d[col] = d.groupby('date')[col].transform(lambda s: s.clip(s.quantile(0.005),s.quantile(0.995)))
    d['r_tail'] = d.groupby('date')['tail'].transform(xr)
    for b in BARRA:
        d[f'r_{b}'] = d.groupby('date')[b].transform(xr)

    # Crowd residual -> IndNeut -> BARRA (3-stage)
    d['crowd_bs'] = d.groupby('date')['BTOP'].transform(lambda s: xr(s.abs()))
    d['_r454a'] = d['r_tail'] - 0.3*d['crowd_bs']
    d['_r454a'] = d['_r454a'].clip(-0.5,0.5)
    d['_z454a'] = d.groupby('date')['_r454a'].transform(crz)
    d['_ind454'] = d.groupby(['date','industry_level1_code'])['_z454a'].transform(crz)
    d['_r454b'] = d.groupby('date')['_ind454'].transform(xr)
    d['_r454b'] = d['_r454b'] * (d['_r454b'].abs() > 0.3).astype(float)
    d['_z454b'] = d.groupby('date')['_r454b'].transform(crz)
    d['_c454'] = bc(d, '_z454b')
    d['factor'] = d.groupby('date')['_c454'].transform(crz)

    out = d.loc[d['date'] >= pd.to_datetime(start_date).normalize(),
                ['date','instrument','factor']].copy()
    out = out.replace([np.inf,-np.inf],np.nan).dropna(subset=['factor'])
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={'date':[start_date,end_date]}).df()
    return pd.merge(out, stk, how='inner', on=['date','instrument']).reset_index(drop=True)

if __name__ == '__main__':
    import structlog; logger = structlog.get_logger()
    ds = {'bar1m': 'bigalpha_2026_stock_bar1m', 'exposure': 'bigalpha_2026_exposure'}
    fd = main(ds, '2024-06-01 00:00:00', '2024-06-30 23:59:59')
    print(fd.head()); print(f"rows={len(fd)}")
    cv = fd.groupby('date').size() / fd['instrument'].nunique()
    print(f"cov min={cv.min():.2%}"); assert cv.min() > 0.6; print('v454 OK')
